# 06 - SASRec (Self-Attentive Sequential Recommendation)

SASRec uses **causal self-attention** over the city sequence: at each step it learns which earlier stops matter most for predicting the next, rather than compressing everything into one state (GRU4Rec) or reading only the last city (Markov / FPMC). It is the natural place to add side features.

For the benchmark it is trained with **full-catalogue cross-entropy** and **AdamW**, the same objective and optimiser as GRU4Rec, so the GRU4Rec-vs-SASRec comparison isolates recurrence vs attention (the original paper's BPR loss is still available via `loss="bpr"`).

### Benchmark protocol (same as GRU4Rec / Markov)
1. **Tune** with seed 42: train on Train, select on validation **NDCG@10** -- 25 Optuna TPE trials (median pruner), full-catalogue cross-entropy, AdamW.
2. **Record $E_{best}$**, the peak epoch of the winning trial.
3. **Merge** Train + Validation.
4. **Train 5 fresh models** (seeds `[1, 7, 13, 2024, 99]`) for exactly $E_{best}$ epochs, no early stopping.
5. **Score on Test**; log **mean** and **std** across seeds to `results.csv`.

**Search space** (the unified space): `embed_dim` {64,128,256}, `num_blocks` {1,2}, `dropout` [0,0.5], `lr` [1e-4,1e-2] log, `weight_decay` [1e-6,1e-3] log, `batch_size` {64,128,256}, `max_len` {10,20,30,40} (same values as GRU4Rec; trips are short, so longer windows are identical). `n_heads` is fixed at 2.

This is done three times -- **no side features**, side features **summed**, side features **concatenated** -- each with its own search and the identical budget.

All logic is in `src/models/sasrec.py` (a hand-rolled transformer so the causal + padding masks are fully controlled). Running this notebook top to bottom does everything and ends by displaying `results.csv`.

> **Runtime:** 3 variants x 25 trials, then 3 x 5 final fits. Use a GPU. To smoke-test the pipeline first, set `N_TRIALS = 2` and `SEEDS = SEEDS[:2]` in the config cell below -- and don't trust those numbers.

In [ ]:
import torch

import os, sys, json
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd
from src.preprocessing import load_processed, preprocess
from src.evaluate import append_result
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

RESULTS = os.path.join(ROOT, 'results.csv')
TUNING_DIR = os.path.join(ROOT, 'results', 'tuning'); os.makedirs(TUNING_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():          # Apple Silicon GPU
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('device:', DEVICE)

In [ ]:
from src.models.sasrec import tune_sasrec as TUNE, final_sasrec as FINAL, SIDE_FEATURES
from src.models.gru4rec import N_TRIALS, FINAL_SEEDS, MAX_EPOCHS, PATIENCE
SEEDS = list(FINAL_SEEDS)
# N_TRIALS = 2; SEEDS = SEEDS[:2]        # <- uncomment for a quick smoke test only
print(f"trials/variant={N_TRIALS} | max_epochs={MAX_EPOCHS} | patience={PATIENCE} | seeds={SEEDS}")

## Ablation - side features on/off, and how they combine

We compare (a) no side features, (b) side features **summed** into the city embedding, and (c) side features **concatenated** then projected back to the model width. The features are trip-level context: `device_class`, `booker_country`, `affiliate_id`. This isolates whether context moves the metrics and whether sum or concat is better.

For each variant: **tune** (validation only) -> read $E_{best}$ -> **final fits** on Train + Val for 5 seeds -> append the mean row and the std row to `results.csv`, and save the per-seed table and chosen hyper-parameters under `results/tuning/`.

Rows written: `SASRec-no-side`, `SASRec-side-sum`, `SASRec-side-concat`, each with a `(std)` row.

In [ ]:
def run_variant(tag, row_name, tune_kw):
    """Steps 1-5 for one variant: tune on validation, read E_best, 5-seed final
    fits on train+val, save params + per-seed table, log mean and std rows."""
    print(f"\n==================== {row_name} ====================")

    # steps 1-2: search on validation, record E_best
    study = TUNE(data, n_trials=N_TRIALS, device=DEVICE, show_progress_bar=False, **tune_kw)
    best_params = dict(study.best_params)
    e_best = study.best_trial.user_attrs["best_epoch"]
    with open(os.path.join(TUNING_DIR, f"{tag}.json"), "w") as f:
        json.dump({"best_params": best_params, "E_best": e_best,
                   "best_val_NDCG@10": study.best_value, "n_trials": len(study.trials)}, f, indent=2)

    # steps 3-5: train+val, 5 seeds x E_best epochs, score on test
    per_seed, _ = FINAL(data, best_params, epochs=e_best, seeds=SEEDS, device=DEVICE, **tune_kw)
    per_seed.to_csv(os.path.join(TUNING_DIR, f"{tag}-per-seed.csv"))

    mean, std = per_seed.mean(), per_seed.std()
    mean_row = {k: (int(v) if k == "n_params" else float(v)) for k, v in mean.items()}
    std_row = {k: float(v) for k, v in std.items()}
    append_result(RESULTS, row_name, mean_row)
    append_result(RESULTS, f"{row_name} (std)", std_row)
    summaries[row_name] = {"best_params": best_params, "E_best": e_best,
                           "val_NDCG@10": round(study.best_value, 4),
                           "test_NDCG@10": f"{mean['NDCG@10']:.4f} +/- {std['NDCG@10']:.4f}"}

summaries = {}

### No side features

In [ ]:
run_variant("sasrec-no-side", "SASRec-no-side", {"side_feats": ()})

### Side features - sum

In [ ]:
run_variant("sasrec-side-sum", "SASRec-side-sum", {"side_feats": SIDE_FEATURES, "side_mode": "sum"})

### Side features - concat

In [ ]:
run_variant("sasrec-side-concat", "SASRec-side-concat", {"side_feats": SIDE_FEATURES, "side_mode": "concat"})

## Selected configurations

In [ ]:
display(pd.DataFrame(summaries).T)

## Final results (`results.csv`)

In [ ]:
display(pd.read_csv(RESULTS))

## What to write up

- Report the side-feature ablation (none vs sum vs concat) as mean +/- std over 5 seeds, next to GRU4Rec's side / no-side pair. A gap smaller than the std is not a real effect; if context doesn't help either neural model, that is a finding about the dataset.
- GRU4Rec vs SASRec now share loss, optimiser, search space and budget, so their gap is recurrence vs attention.
- Contrast SASRec with FPMC/Markov by **trip length** (the comparison notebook slices this): the attention advantage should show up on longer trips.
- Compute: see `n_params` / `train_secs` in the per-seed CSVs for the accuracy-vs-cost discussion.